In [1]:
import pandas as pd
pd.DataFrame.iteritems = pd.DataFrame.items
import numpy as np
import os
import re
import pickle

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits import mplot3d
import matplotlib.ticker as mticker
import matplotlib.transforms as transforms
import matplotlib.patches as patches
import matplotlib.cm as cm

import plotly.express as px
import scipy

from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import PCA, KernelPCA

from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.svm import SVC, SVR
from sklearn.neural_network import MLPClassifier
from sklearn import metrics
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
filename = "Preprocessed_Raman_Spectra_2025.pkl"
with open(filename, "rb") as f:
    loaded_data = pickle.load(f)
    
print(loaded_data['Spectra'].shape)

(360, 1003)


In [3]:
loaded_data["Preprocess conditions"]

{'order': 'SI --> CS --> BSR --> SG --> ED',
 'CS': [1, 0.01],
 'BSR': [100, 1],
 'SG': [15, 2],
 'ED': [400, 1800]}

In [4]:
wavenumber = loaded_data["wavenumber"]
spectral = loaded_data["Spectra"]

bacteria_species = loaded_data["species label"]
culture_age = loaded_data['culture age']
cul_cond = loaded_data["culture condition"]

biological_rep = loaded_data["biological replicate"]
technical_rep = loaded_data["technical replicate"]

In [5]:
condition_dict = {"A" : "0 mM benzoate", "C" : "20 mM benzoate"}
culture_condition = []
for i in range(len(cul_cond)):
    culture_condition.append(condition_dict[cul_cond[i]])
culture_condition = np.array(culture_condition)

# Biological Replicate Seperation

In [6]:
label_species = np.unique(bacteria_species)
label_condition = np.unique(culture_condition)
label_age = np.unique(culture_age)

species_dict = {}
for i in range(len(label_species)):
    species_dict[label_species[i]] = i
    
condition_dict = {}
for i in range(len(label_condition)):
    condition_dict[label_condition[i]] = i
    
age_dict = {}
for i in range(len(label_age)):
    age_dict[label_age[i]] = i
    
y_output = []
for i in range(len(bacteria_species)):
    temp = [species_dict[bacteria_species[i]],
           condition_dict[culture_condition[i]],
            age_dict[culture_age[i]]
           ]
    y_output.append(np.array(temp))

y_data_full = np.array(y_output)
X_data_full = spectral

print(X_data_full.shape, y_data_full.shape)

(360, 1003) (360, 3)


In [7]:
def nCV_eval(train_X, test_X, train_y, test_y):
    names = ["species", "metabolites", "age"]
    overall_performance = {}
    rs = 42

    for j in range(len(names)):

        X_data_train = train_X
        y_data_train = train_y[:,j]

        X_data_test = test_X
        y_data_test = test_y[:,j]

        print(names[j], X_data_train.shape, y_data_train.shape)
        
        param_grid = { "kernel": ["linear"],
                      "C" : np.logspace(-3, 3, 8)}

        SVC_model = SVC(random_state = 42)

        SVC_grid = GridSearchCV(SVC_model, 
                                 param_grid, 
                                 n_jobs = -1, 
                                 verbose = 0, 
                                 return_train_score = True)

        SVC_grid.fit(X_data_train, y_data_train)

        SVC_best_model = SVC_grid.best_estimator_
        y_predict = SVC_best_model.predict(X_data_test)

        report = classification_report(y_data_test, y_predict, output_dict = True)
        con_mat = confusion_matrix(y_data_test, y_predict)

        overall_performance[names[j]] = [X_data_train, y_data_train, X_data_test, y_data_test, 
                                         y_predict, SVC_grid, SVC_best_model, con_mat, report]

    return overall_performance

In [8]:
import time

uni_br = np.unique(biological_rep)
NCV_performance = {}

for i in range(len(uni_br)):
    strcv = "CV " + str(uni_br[i])
    
    print(strcv)
    
    ind_train = biological_rep != uni_br[i]
    ind_test = biological_rep == uni_br[i]
    
    train_X_full_us = X_data_full[ind_train,:]
    test_X_full_us = X_data_full[ind_test,:]
    
    SS_overall = StandardScaler()
    
    train_X_full = SS_overall.fit_transform(train_X_full_us)
    test_X_full = SS_overall.transform(test_X_full_us)
        
    train_y_data = y_data_full[ind_train,:]
    test_y_data = y_data_full[ind_test,:]
    
        
    temp = nCV_eval(train_X_full,
                    test_X_full, 
                    train_y_data, 
                    test_y_data)


    for n in list(temp.keys()):
        print(n + " accuracy: ", round(temp[n][-1]["accuracy"], 2))

    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
        
    NCV_performance[strcv] = temp
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

CV 1
species (240, 1003) (240,)
metabolites (240, 1003) (240,)
age (240, 1003) (240,)
species accuracy:  0.94
metabolites accuracy:  0.76
age accuracy:  0.85
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
CV 2
species (240, 1003) (240,)
metabolites (240, 1003) (240,)
age (240, 1003) (240,)
species accuracy:  0.88
metabolites accuracy:  0.72
age accuracy:  0.85
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
CV 3
species (240, 1003) (240,)
metabolites (240, 1003) (240,)
age (240, 1003) (240,)
species accuracy:  0.85
metabolites accuracy:  0.73
age accuracy:  0.8
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [9]:
import pickle

my_dictionary = NCV_performance
 
# Open a file in write binary mode ('wb')
with open('Performance Dictionaries\\Performance_SVC_nCV.pkl', 'wb') as file:
    # Serialize the dictionary and write it to the file
    pickle.dump(my_dictionary, file)